# 03 Curate, build arms, freeze test

Runs the curation gates (our deterministic gates and verification, then CuratorKIT dedup / grounding / reward)
on `data/raw/`, writing `data/curated/{split}.jsonl` and a per-row decision log `data/curated/gate_log.jsonl`.
Then builds the A1 (raw) and A2 (curated) training sets at equal row count and strata under `data/arms/`, writes
`data/test/review.csv` for the hand check, and freezes `data/test/test.jsonl` with `data/test/test.sha256`.
Manifest: none yet; the frozen test set is pinned by its sha256 instead, and `evals.run_eval` refuses a mismatch.

**Needs:** `JUDGE_MODEL` (non-Cohere) and its provider key for the hallucination gate (skipped if unset). CPU is fine.
**Manual step:** hand-write the adversarial rows in `evals/adversarial_test.jsonl` and fill `keep=y/n` in
`data/test/review.csv` before the freeze cell.

In [ ]:
import json, os, sys
from collections import Counter
from pathlib import Path

# Run from the repo root so relative paths match the README runbook.
ROOT = Path.cwd()
while not (ROOT / "core").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
PY = sys.executable  # `!{PY} -m ...` runs the CLI with this kernel's interpreter
print("repo root:", ROOT)

In [ ]:
INDEX = "index/rbi-2026-11-10"
if not os.environ.get("JUDGE_MODEL"):
    print("WARNING: JUDGE_MODEL unset; the grounding/hallucination gate is skipped (rows marked n/a)")

In [ ]:
from pipeline.curate import curate

curate(Path("data/raw"), Path("corpus"), Path("data/curated"))

In [ ]:
!{PY} -m pipeline.build_arms arms

In [ ]:
!{PY} -m pipeline.build_arms review

**Stop here.** Fill `keep=y/n` in `data/test/review.csv`, then run the freeze cell.

In [ ]:
!{PY} -m pipeline.build_arms freeze --index {INDEX}

## Summary

In [ ]:
def read_jsonl(path):
    path = Path(path)
    return [json.loads(l) for l in path.open(encoding="utf-8")] if path.exists() else []


for split in ("train", "dev", "test"):
    raw, cur = read_jsonl(f"data/raw/{split}.jsonl"), read_jsonl(f"data/curated/{split}.jsonl")
    print(f"{split}: {len(raw)} raw -> {len(cur)} curated")
log = read_jsonl("data/curated/gate_log.jsonl")
print("gate decisions:", dict(Counter((e["gate"], e["decision"]) for e in log)))
for arm in ("A1", "A2"):
    rows = read_jsonl(f"data/arms/{arm}/rows.jsonl")
    print(f"{arm}: {len(rows)} rows", dict(Counter((r["kind"], r["lang"]) for r in rows)))
test = read_jsonl("data/test/test.jsonl")
sha = Path("data/test/test.sha256")
print(f"frozen test: {len(test)} rows, sha256 {sha.read_text().strip() if sha.exists() else '(not frozen)'}")